In [191]:
import pandas as pd

In [192]:
df = pd.read_csv("predictive_maintenance.csv")
df

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure
...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,No Failure
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,No Failure
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,No Failure
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,No Failure


In [193]:
df = df.drop(columns = ['UDI',"Product ID", "Failure Type"])

In [194]:
df['Target'].value_counts()
from sklearn.model_selection import train_test_split

In [195]:
x = df.drop(columns = 'Target')
y = df['Target']
x_train , x_test , y_train , y_test = train_test_split(x,y, test_size = 0.2 , random_state = 42)

In [196]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [197]:
trf1 = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(sparse_output = False , handle_unknown = 'ignore'),['Type'])
], remainder = 'passthrough')

In [198]:
trf1.set_output(transform='pandas')

,transformers,"[('ohe', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [199]:
x_train = trf1.fit_transform(x_train)
x_test = trf1.transform(x_test)

In [200]:
x_train

,ohe__Type_H,ohe__Type_L,ohe__Type_M,remainder__Air temperature [K],remainder__Process temperature [K],remainder__Rotational speed [rpm],remainder__Torque [Nm],remainder__Tool wear [min]
9254,0.0,1.0,0.0,298.3,309.1,1616,31.1,195
1561,0.0,1.0,0.0,298.2,308.4,1388,53.8,137
1670,0.0,1.0,0.0,298.2,307.8,1528,31.1,194
6087,0.0,0.0,1.0,300.9,310.8,1599,33.0,7
6669,0.0,1.0,0.0,301.4,310.5,1571,33.9,208
...,...,...,...,...,...,...,...,...
5734,0.0,1.0,0.0,302.3,311.8,1369,56.2,208
5191,0.0,1.0,0.0,304.0,313.2,1416,46.0,128
5390,1.0,0.0,0.0,302.8,312.3,1483,47.2,223
860,1.0,0.0,0.0,296.1,306.9,1541,32.6,33


In [201]:
ss = StandardScaler().set_output(transform='pandas')
x_train = ss.fit_transform(x_train)
x_test = ss.transform(x_test)

In [202]:
x_train

,ohe__Type_H,ohe__Type_L,ohe__Type_M,remainder__Air temperature [K],remainder__Process temperature [K],remainder__Rotational speed [rpm],remainder__Torque [Nm],remainder__Tool wear [min]
9254,-0.333796,0.821609,-0.658943,-0.854066,-0.609589,0.427634,-0.892696,1.375035
1561,-0.333796,0.821609,-0.658943,-0.904014,-1.080528,-0.834945,1.382187,0.457620
1670,-0.333796,0.821609,-0.658943,-0.904014,-1.484190,-0.059677,-0.892696,1.359218
6087,-0.333796,-1.217123,1.517582,0.444571,0.534121,0.333495,-0.702288,-1.598655
6669,-0.333796,0.821609,-0.658943,0.694309,0.332290,0.178441,-0.612094,1.580663
...,...,...,...,...,...,...,...,...
5734,-0.333796,0.821609,-0.658943,1.143837,1.206891,-0.940159,1.622704,1.580663
5191,-0.333796,0.821609,-0.658943,1.992946,2.148770,-0.679891,0.600509,0.315263
5390,2.995841,-1.217123,-0.658943,1.393575,1.543276,-0.308870,0.720767,1.817925
860,2.995841,-1.217123,-0.658943,-1.952913,-2.089684,0.012312,-0.742374,-1.187400


In [203]:
from imblearn.under_sampling import RandomUnderSampler

In [204]:
rr = RandomUnderSampler(random_state=42)
u_x_train , u_y_train = rr.fit_resample(x_train , y_train)
u_x_test , u_y_test = rr.fit_resample(x_test , y_test)

In [211]:
from xgboost import XGBClassifier


In [210]:
import re
u_x_train.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in u_x_train.columns]
u_x_test.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in u_x_test.columns]

In [213]:
xg = XGBClassifier(n_estimators = 50)
xg.fit(u_x_train,u_y_train)
xg_pred = xg.predict(u_x_test)

In [215]:
from sklearn.metrics import accuracy_score, classification_report

In [216]:
accuracy_score(u_y_test, xg_pred)

0.9098360655737705

In [218]:
print(classification_report(u_y_test, xg_pred))

              precision    recall  f1-score   support

           0       0.95      0.87      0.91        61
           1       0.88      0.95      0.91        61

    accuracy                           0.91       122
   macro avg       0.91      0.91      0.91       122
weighted avg       0.91      0.91      0.91       122



In [223]:
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE

In [224]:
s = SMOTE()
s_x_train , s_y_train = s.fit_resample(x_train,y_train)
s_x_test , s_y_test = s.fit_resample(x_test, y_test)

In [226]:
brf  = BalancedRandomForestClassifier(n_estimators=50)
brf.fit(s_x_train, s_y_train)
brf_pred = brf.predict(s_x_test)
accuracy_score(s_y_test,brf_pred)

0.8780299123259412

In [229]:
print(classification_report(s_y_test , brf_pred))

              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1939
           1       0.97      0.78      0.87      1939

    accuracy                           0.88      3878
   macro avg       0.89      0.88      0.88      3878
weighted avg       0.89      0.88      0.88      3878



In [230]:
import joblib

In [233]:
pipe_line = {
    'transfomer': trf1,
    'scaler': ss,
    'xgboost': xg
}

In [235]:
joblib.dump(pipe_line , "machine_failure_prediction.joblib")

['machine_failure_prediction.joblib']